In [11]:
import re
import csv
import os

# 設定路徑
input_file = "data.txt"
output_file = "/Users/hungwei/Desktop/Proj/Mamba3-XR/pre-train/colab_training_log.csv"

# --- 更新後的 Regex Pattern ---
# 針對你的格式優化：支援行首時間戳，並精確捕捉各個數值
pattern = re.compile(
    r"(?P<elapsed_s>[\d\.]+)s\s+\d+\s+step\s+(?P<step>\d+)/\d+\s*\|\s*"  # 行首時間與 Step
    r"Loss:\s+(?P<loss>[\d\.]+)\s*"                                     # Total Loss
    r"\(CE:(?P<ce_loss>[\d\.]+),\s*"                                    # CE Loss
    r"LB:(?P<lb_contrib>[\d\.]+),\s*"                                   # LB Contrib
    r"Z:(?P<z_contrib>[\d\.]+)\)\s*\|\s*"                               # Z Contrib
    r"PPL:\s+[\d\.]+\s*\|\s*"                                           # PPL (略過)
    r"Grad:\s+(?P<grad_norm>[\d\.]+)\s*\|\s*"                           # Grad Norm
    r"T_router:\s+(?P<router_temp>[\d\.]+)\s*\|\s*"                     # Router Temp
    r"LR:\s+(?P<lr>[\d\.e\-]+)\s*\|\s*"                                 # Learning Rate
    r"Time:\s+(?P<step_time_s>[\d\.]+)s"                                # Step Time
)

def safe_float(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        return 0.0

log_data = []

if not os.path.exists(input_file):
    print(f"❌ 錯誤：找不到輸入檔案 '{input_file}'")
else:
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                match = pattern.search(line)
                if match:
                    data = match.groupdict()
                    # 按照你要求的格式排序
                    row = {
                        'step': int(data['step']),
                        'loss': safe_float(data['loss']),
                        'ce_loss': safe_float(data['ce_loss']),
                        'lb_contrib': safe_float(data['lb_contrib']),
                        'z_contrib': safe_float(data['z_contrib']),
                        'router_temp': safe_float(data['router_temp']),
                        'lr': safe_float(data['lr']),
                        'grad_norm': safe_float(data['grad_norm']),
                        'loss_scale': 0.0,    # Log 中未發現此欄位，暫設 0
                        'tokens_seen': 0,     # Log 中未發現此欄位，暫設 0
                        'elapsed_s': safe_float(data['elapsed_s']),
                        'step_time_s': safe_float(data['step_time_s'])
                    }
                    log_data.append(row)
        
        print(f"🔍 掃描完成，找到 {len(log_data)} 筆符合格式的資料。")
    except Exception as e:
        print(f"❌ 讀取時發生錯誤: {e}")

# --- 寫入 CSV ---
if log_data:
    # 依照 step 排序
    log_data.sort(key=lambda x: x['step'])
    
    header = [
        'step', 'loss', 'ce_loss', 'lb_contrib', 'z_contrib', 
        'router_temp', 'lr', 'grad_norm', 'loss_scale', 
        'tokens_seen', 'elapsed_s', 'step_time_s'
    ]

    try:
        output_dir = os.path.dirname(output_file)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)

        # 這裡改為 'w' (複寫) 或 'a' (追加)，依據你的需求。
        # 如果要保留舊資料請用 'a'，如果要全新產生請用 'w'。
        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=header)
            writer.writeheader()
            writer.writerows(log_data)
        
        print(f"✅ 成功寫入至: {output_file}")
    except Exception as e:
        print(f"❌ 寫入 CSV 失敗: {e}")

🔍 掃描完成，找到 5637 筆符合格式的資料。
✅ 成功寫入至: /Users/hungwei/Desktop/Proj/Mamba3-XR/pre-train/colab_training_log.csv


In [12]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from pathlib import Path

# ==========================================
# Mamba-3-XR Training Log Analyzer 
# (Compatible with: step,loss,ce_loss,lb_contrib,z_contrib,router_temp,lr,grad_norm,loss_scale,tokens_seen,elapsed_s,step_time_s)
# ==========================================

class TrainingAnalyzer:
    def __init__(self, csv_path, platform_info="NVIDIA A100", global_batch=36, seq_len=512, moving_window=10):
        self.csv_path = Path(csv_path)
        self.platform_info = platform_info
        self.global_batch = global_batch
        self.seq_len = seq_len
        self.moving_window = moving_window
        
        if not self.csv_path.exists():
            raise FileNotFoundError(f"Error: Log file not found at {self.csv_path}")
            
        self.df = self._load_and_preprocess()
        
    def _load_and_preprocess(self):
        df = pd.read_csv(self.csv_path)
        
        # 自動去除所有欄位名稱前後的空白字元
        df.columns = df.columns.str.strip()
        
        # 防呆機制：檢查主要的 loss 欄位是否存在
        if 'loss' not in df.columns:
            print("❌ 當前讀取到的欄位名稱有：", df.columns.tolist())
            raise KeyError("找不到 'loss'，請檢查上方印出的欄位名稱是否包含錯字或未換行。")
            
        # 排序並移除重複
        df = df.sort_values("step").drop_duplicates(subset=['step']).reset_index(drop=True)
        
        # 累計時間 (小時)
        if 'elapsed_s' in df.columns:
            df["cum_time_hrs"] = df["elapsed_s"] / 3600
        else:
            df["cum_time_hrs"] = df["step_time_s"].cumsum() / 3600
            
        # 動態吞吐量 (Tokens/Sec)
        df["tokens_per_sec"] = (self.global_batch * self.seq_len) / df["step_time_s"]
        
        # ==========================================
        # 新增：透過 CE Loss 計算 PPL (Perplexity)
        # ==========================================
        if 'ce_loss' in df.columns:
            # 使用 np.exp 計算，並加上 clip 避免初期 loss 過高導致數值溢位
            df["ppl"] = np.exp(np.clip(df["ce_loss"], None, 20))
        
        # 新增需要平滑處理的欄位清單 (加入 ppl)
        metrics_to_smooth = [
            "loss", "ce_loss", "ppl", "lb_contrib", "z_contrib", 
            "router_temp", "grad_norm", "loss_scale", "tokens_per_sec"
        ]
        
        for col in metrics_to_smooth:
            if col in df.columns:
                df[f"{col}_ma"] = df[col].rolling(self.moving_window, min_periods=1).mean()
                
        return df
    
    def generate_report(self, export_html=True, output_filename="mamba3_training_report.html"):
        df = self.df
        
        # 升級為 4 列版面：前三列 3x3 網格，第四列貫穿左右放吞吐量
        fig = make_subplots(
            rows=4, cols=3,
            specs=[
                [{}, {}, {}],
                [{}, {}, {}],
                [{}, {}, {}],
                [{"colspan": 3}, None, None]
            ],
            subplot_titles=(
                "Total Loss", "Cross Entropy Loss", "Perplexity (PPL)", 
                "Load Balancing Contrib", "Z-Loss Contrib", "Router Temperature",
                "Learning Rate", "Gradient Norm", "Loss Scale", 
                "Throughput (Tokens/Sec)"
            ),
            vertical_spacing=0.08,
            horizontal_spacing=0.08,
            shared_xaxes=True
        )

        def add_metric(fig, x, y, y_ma, name, color, row, col, unit="", show_raw=True):
            if show_raw:
                fig.add_trace(go.Scatter(x=x, y=y, name=f"{name} (Raw)", 
                                         line=dict(color=color, width=1), opacity=0.2,
                                         showlegend=False, hoverinfo='skip'), row=row, col=col)
            fig.add_trace(go.Scatter(x=x, y=y_ma, name=f"{name}", 
                                     line=dict(color=color, width=2.5),
                                     hovertemplate=f'Step: %{{x}}<br>{name}: %{{y:.4f}}{unit}'), row=row, col=col)

        # 配色方案 (新增 PPL 專屬顏色)
        c_loss  = "#1f77b4" # Blue
        c_ce    = "#2ca02c" # Green
        c_ppl   = "#d62728" # Red (PPL)
        c_lb    = "#ff7f0e" # Orange
        c_z     = "#9467bd" # Purple
        c_lr    = "#e377c2" # Pink
        c_grad  = "#8c564b" # Brown 
        c_temp  = "#bcbd22" # Olive
        c_scale = "#7f7f7f" # Gray
        c_tps   = "#17becf" # Cyan

        # --- Row 1 ---
        add_metric(fig, df['step'], df['loss'], df['loss_ma'], "Total Loss", c_loss, 1, 1)
        add_metric(fig, df['step'], df['ce_loss'], df['ce_loss_ma'], "CE Loss", c_ce, 1, 2)
        add_metric(fig, df['step'], df['ppl'], df['ppl_ma'], "PPL", c_ppl, 1, 3)

        # --- Row 2 ---
        add_metric(fig, df['step'], df['lb_contrib'], df['lb_contrib_ma'], "LB Contrib", c_lb, 2, 1)
        add_metric(fig, df['step'], df['z_contrib'], df['z_contrib_ma'], "Z-Loss Contrib", c_z, 2, 2)
        add_metric(fig, df['step'], df['router_temp'], df['router_temp_ma'], "Router Temp", c_temp, 2, 3)

        # --- Row 3 ---
        fig.add_trace(go.Scatter(x=df['step'], y=df['lr'], name="LR", 
                                 line=dict(color=c_lr, width=2.5),
                                 hovertemplate='Step: %{x}<br>LR: %{y:.2e}'), row=3, col=1)
        add_metric(fig, df['step'], df['grad_norm'], df['grad_norm_ma'], "Gradient Norm", c_grad, 3, 2)
        add_metric(fig, df['step'], df['loss_scale'], df['loss_scale_ma'], "Loss Scale", c_scale, 3, 3)

        # --- Row 4 (貫穿整列) ---
        add_metric(fig, df['step'], df['tokens_per_sec'], df['tokens_per_sec_ma'], "Throughput", c_tps, 4, 1, " T/s")

        # 自動計算合適的 dtick (避免標籤重疊)
        max_step = df['step'].max()
        suggested_dtick = 50 if max_step <= 500 else (max_step // 10)

        # ==========================================
        # 論文專業格式設定區塊
        # ==========================================
        fig.update_layout(
            height=1300, # 配合 4 列調高版面
            title_x=0.5, 
            template="simple_white", 
            font=dict(
                family="Times New Roman, Times, serif", 
                size=14, 
                color="black"
            ),
            hovermode="x unified",
            legend=dict(
                orientation="h", 
                yanchor="bottom", y=1.02, 
                xanchor="right", x=1,
                bordercolor="black",
                borderwidth=1,
                font=dict(size=13)
            ),
            plot_bgcolor='white',
            paper_bgcolor='white'
        )
        
        fig.update_xaxes(
            showline=True, linewidth=1.5, linecolor='black', mirror=True, 
            gridcolor='rgba(200,200,200,0.3)', 
            ticks="outside", tickwidth=1.5, ticklen=6, tickcolor='black',
            tickmode='linear', dtick=suggested_dtick,
            showticklabels=True, matches='x',
            title_font=dict(size=14, family="Times New Roman")
        )
        
        fig.update_yaxes(
            showline=True, linewidth=1.5, linecolor='black', mirror=True, 
            gridcolor='rgba(200,200,200,0.3)', 
            ticks="outside", tickwidth=1.5, ticklen=6, tickcolor='black',
            title_font=dict(size=14, family="Times New Roman")
        )
        
        for annotation in fig['layout']['annotations']: 
            annotation['font'] = dict(size=16, family="Times New Roman, Times, serif")

        fig.show()
        if export_html:
            fig.write_html(output_filename)
            print(f"✅ Report saved: {output_filename}")

    def print_summary(self):
        df = self.df
        last = df.iloc[-1]
        
        print(f"\n{'='*60}")
        print(f"         MAMBA-3-XR LATEST STATUS (Step {last['step']})")
        print(f"{'='*60}")
        print(f"Time Elapsed    : {last['cum_time_hrs']:.2f} Hours")
        print(f"Throughput      : {last['tokens_per_sec_ma']:.2f} Tokens/s")
        if 'tokens_seen' in df.columns:
            print(f"Tokens Seen     : {last['tokens_seen']:.2e}")
        print(f"Learning Rate   : {last['lr']:.2e}")
        print(f"Gradient Norm   : {last['grad_norm_ma']:.4f}") 
        print(f"Loss Scale      : {last['loss_scale_ma']:.2f}")
        print(f"Router Temp     : {last['router_temp_ma']:.4f}")
        print(f"{'-'*60}")
        print(f"Total Loss      : {last['loss_ma']:.4f}")
        print(f"CE Loss         : {last['ce_loss_ma']:.4f}")
        if 'ppl_ma' in last:
            print(f"PPL             : {last['ppl_ma']:.2f}")
        print(f"LB Contrib      : {last['lb_contrib_ma']:.4f}")
        print(f"Z Contrib       : {last['z_contrib_ma']:.4f}")
        print(f"{'='*60}\n")

if __name__ == "__main__":
    # 確保路徑與前一個腳本輸出的位置一致
    LOG_PATH = "/Users/hungwei/Desktop/Proj/Mamba3-XR/pre-train/colab_training_log.csv" 
    
    try:
        analyzer = TrainingAnalyzer(LOG_PATH)
        analyzer.generate_report()
        analyzer.print_summary()
    except Exception as e:
        print(f"❌ Error: {e}")

✅ Report saved: mamba3_training_report.html

         MAMBA-3-XR LATEST STATUS (Step 18088.0)
Time Elapsed    : 10.93 Hours
Throughput      : 780.89 Tokens/s
Tokens Seen     : 0.00e+00
Learning Rate   : 7.13e-05
Gradient Norm   : 0.6540
Loss Scale      : 0.00
Router Temp     : 0.5000
------------------------------------------------------------
Total Loss      : 4.9285
CE Loss         : 4.7269
PPL             : 113.84
LB Contrib      : 0.2012
Z Contrib       : 0.0004



In [13]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from pathlib import Path

# ==========================================
# Mamba-3-XR Training Log Analyzer 
# (支援 v1/v2/v3 時期三色區隔與論文級排版 - 支援動態 Seq Length & Global Batch)
# ==========================================

class TrainingAnalyzer:
    def __init__(self, v3_csv_path, v2_csv_path=None, v1_csv_path=None, platform_info="NVIDIA A100", 
                 v1_global_batch=36, v2_global_batch=32, v3_global_batch=32,
                 v1_seq_len=512, v2_seq_len=512, v3_seq_len=1204, moving_window=10):
        self.v3_csv_path = Path(v3_csv_path)
        self.v2_csv_path = Path(v2_csv_path) if v2_csv_path else None
        self.v1_csv_path = Path(v1_csv_path) if v1_csv_path else None
        
        self.platform_info = platform_info
        
        # 紀錄各版本的 Global Batch Size
        self.v1_global_batch = v1_global_batch
        self.v2_global_batch = v2_global_batch
        self.v3_global_batch = v3_global_batch
        
        # 紀錄各版本的 Sequence Length
        self.v1_seq_len = v1_seq_len
        self.v2_seq_len = v2_seq_len
        self.v3_seq_len = v3_seq_len
        
        self.moving_window = moving_window
        self.split_token_marks = [] # 記錄 v1->v2 和 v2->v3 的交界點 (Token 數)
        
        if not self.v3_csv_path.exists():
            raise FileNotFoundError(f"Error: Current v3 log file not found at {self.v3_csv_path}")
            
        self.df_curr, self.df_combined = self._load_and_preprocess()
        
    def _load_and_preprocess(self):
        dfs = []
        current_abs_step_offset = 0
        current_total_tokens_offset = 0  
        
        def process_historical_df(path, version_name, current_seq_len, current_global_batch):
            nonlocal current_abs_step_offset, current_total_tokens_offset
            if path and path.exists():
                df = pd.read_csv(path)
                df.columns = df.columns.str.strip()
                df['version'] = version_name
                
                # 兼容舊版欄位名稱
                if 'loss' in df.columns and 'total_loss' not in df.columns:
                    df = df.rename(columns={'loss': 'total_loss'})
                if 'ce_loss' not in df.columns and 'aux_loss' in df.columns:
                    df['ce_loss'] = df['total_loss'] - df['aux_loss']
                    
                df = df.sort_values("step").drop_duplicates(subset=['step']).reset_index(drop=True)
                
                # 計算該版本的 Tokens per step
                tokens_per_step = current_global_batch * current_seq_len
                
                # 計算絕對步數與累積 Tokens
                df['abs_step'] = df['step'] + current_abs_step_offset
                df['total_tokens'] = df['step'] * tokens_per_step + current_total_tokens_offset
                
                # 更新 offset 供下一個版本使用
                current_abs_step_offset = df['abs_step'].max()
                current_total_tokens_offset = df['total_tokens'].max()
                
                # 記錄交界點的 Token 數量
                self.split_token_marks.append(current_total_tokens_offset)
                dfs.append(df)

        # 1. 處理 v1 與 v2 (歷史)，帶入各自的 seq_len 與 global_batch
        process_historical_df(self.v1_csv_path, 'v1', self.v1_seq_len, self.v1_global_batch)
        process_historical_df(self.v2_csv_path, 'v2', self.v2_seq_len, self.v2_global_batch)

        # 2. 處理 v3 (當前)
        df_curr = pd.read_csv(self.v3_csv_path)
        df_curr.columns = df_curr.columns.str.strip()
        df_curr['version'] = 'v3'
        
        if 'total_loss' not in df_curr.columns:
            raise KeyError("找不到 'total_loss'，請檢查 v3 日誌欄位。")
            
        df_curr = df_curr.sort_values("step").drop_duplicates(subset=['step']).reset_index(drop=True)
        
        # 針對 v3 計算 Tokens
        tokens_per_step_v3 = self.v3_global_batch * self.v3_seq_len
        df_curr['abs_step'] = df_curr['step'] + current_abs_step_offset
        df_curr['total_tokens'] = df_curr['step'] * tokens_per_step_v3 + current_total_tokens_offset
        df_curr["tokens_per_sec"] = tokens_per_step_v3 / df_curr["step_time"]
        
        # 平滑處理當前指標 (包含新增的 grad)
        metrics_to_smooth_curr = ["total_loss", "ce_loss", "lb_loss", "z_loss", "ppl", "tokens_per_sec", "mem_gb", "grad"]
        for col in metrics_to_smooth_curr:
            if col in df_curr.columns:
                df_curr[f"{col}_ma"] = df_curr[col].rolling(self.moving_window, min_periods=1).mean()
        
        dfs.append(df_curr)
        df_combined = pd.concat(dfs, ignore_index=True)

        # 3. 計算全局指標平滑
        df_combined['total_loss_ma'] = df_combined['total_loss'].rolling(self.moving_window, min_periods=1).mean()
        df_combined['ppl_ma'] = df_combined['ppl'].rolling(self.moving_window, min_periods=1).mean()
        df_combined['ce_loss_ma'] = df_combined['ce_loss'].rolling(self.moving_window, min_periods=1).mean()
        df_combined["cum_time_hrs"] = df_combined["step_time"].cumsum() / 3600
        
        return df_curr, df_combined
    
    def generate_report(self, export_html=True, output_filename="mamba3_v3_training_report.html"):
        df = self.df_curr
        df_combined = self.df_combined
        
        # 擴充為 7 列
        fig = make_subplots(
            rows=7, cols=2,
            specs=[
                [{}, {}], [{}, {}], [{}, {}], [{}, {}],
                [{"colspan": 2}, None], # Row 5: Grad (大圖)
                [{"colspan": 2}, None], # Row 6: 歷史 Total Loss (大圖)
                [{}, {}]                # Row 7: 歷史 PPL, CE Loss
            ],
            subplot_titles=(
                "Total Loss (Current v3)", "Perplexity (Current v3)", 
                "Cross Entropy Loss (Current v3)", "Load Balancing Loss (Current v3)",
                "Z-Loss (Current v3)", "Learning Rate (Current v3)", 
                "Memory Usage (Current v3)", "Throughput (Current v3)",
                "Gradient Norm (Current v3)",
                "Historical Trend: Total Loss vs Total Tokens (v1 ➜ v2 ➜ v3)",
                "Historical Trend: PPL vs Total Tokens", "Historical Trend: CE Loss vs Total Tokens"
            ),
            vertical_spacing=0.04, horizontal_spacing=0.08
        )

        def add_current_metric(fig, x, y, y_ma, name, color, row, col):
            fig.add_trace(go.Scatter(x=x, y=y, name=f"{name} (Raw)", line=dict(color=color, width=1), opacity=0.2, showlegend=False, hoverinfo='skip'), row=row, col=col)
            fig.add_trace(go.Scatter(x=x, y=y_ma, name=f"{name}", line=dict(color=color, width=2.5), hovertemplate=f'Step: %{{x}}<br>{name}: %{{y:.4f}}'), row=row, col=col)

        def add_split_hist_metric(fig, df, col_name, name, color_map, row, col):
            for version, (c_line, c_fill) in color_map.items():
                v_df = df[df['version'] == version]
                if not v_df.empty:
                    fig.add_trace(go.Scatter(x=v_df['total_tokens'], y=v_df[col_name], name=f"{name} {version} (Raw)", 
                                             line=dict(color=c_line, width=1), opacity=0.2, showlegend=False, hoverinfo='skip'), row=row, col=col)
                    fig.add_trace(go.Scatter(x=v_df['total_tokens'], y=v_df[f"{col_name}_ma"], name=f"{name} ({version})", 
                                             line=dict(color=c_line, width=2.5), fill='tozeroy', fillcolor=c_fill,
                                             hovertemplate=f'Tokens: %{{x:.2e}}<br>{name} {version}: %{{y:.4f}}'), row=row, col=col)
            
            # 加入多段分隔虛線
            for mark in self.split_token_marks:
                fig.add_vline(x=mark, line_dash="dash", line_color="rgba(100,100,100,0.6)", line_width=1.5, row=row, col=col)

        # --- Row 1-4: 當前 v3 訓練狀態 ---
        add_current_metric(fig, df['step'], df['total_loss'], df['total_loss_ma'], "Total Loss", "#1f77b4", 1, 1)
        add_current_metric(fig, df['step'], df['ppl'], df['ppl_ma'], "PPL", "#d62728", 1, 2)
        add_current_metric(fig, df['step'], df['ce_loss'], df['ce_loss_ma'], "CE Loss", "#2ca02c", 2, 1)
        add_current_metric(fig, df['step'], df['lb_loss'], df['lb_loss_ma'], "LB Loss", "#ff7f0e", 2, 2)
        add_current_metric(fig, df['step'], df['z_loss'], df['z_loss_ma'], "Z-Loss", "#9467bd", 3, 1)
        
        fig.add_trace(go.Scatter(x=df['step'], y=df['lr'], name="LR", line=dict(color="#e377c2", width=2.5), hovertemplate='Step: %{x}<br>LR: %{y:.2e}'), row=3, col=2)
        fig.add_trace(go.Scatter(x=df['step'], y=df['mem_gb'], name="Memory", fill='tozeroy', line=dict(color="#7f7f7f", width=1.5), hovertemplate='Step: %{x}<br>Mem: %{y:.2f} GB'), row=4, col=1)
        add_current_metric(fig, df['step'], df['tokens_per_sec'], df['tokens_per_sec_ma'], "Throughput", "#17becf", 4, 2)

        # --- Row 5: Gradient Norm (大橫幅) ---
        if 'grad' in df.columns:
            add_current_metric(fig, df['step'], df['grad'], df['grad_ma'], "Grad Norm", "#8c564b", 5, 1)

        # --- Row 6-7: 歷史整合狀態 (三段漸層色區分 v1, v2, v3) ---
        
        loss_colors = {
            'v1': ("#85C1E9", "rgba(133, 193, 233, 0.1)"),
            'v2': ("#3498DB", "rgba(52, 152, 219, 0.15)"),
            'v3': ("#1B4F72", "rgba(27, 79, 114, 0.2)")
        }
        add_split_hist_metric(fig, df_combined, 'total_loss', "Total Loss", loss_colors, row=6, col=1)
        
        ppl_colors = {
            'v1': ("#F5B7B1", "rgba(245, 183, 177, 0.1)"),
            'v2': ("#E74C3C", "rgba(231, 76, 60, 0.15)"),
            'v3': ("#7B241C", "rgba(123, 36, 28, 0.2)")
        }
        add_split_hist_metric(fig, df_combined, 'ppl', "PPL", ppl_colors, row=7, col=1)
                              
        ce_colors = {
            'v1': ("#A2D9CE", "rgba(162, 217, 206, 0.1)"),
            'v2': ("#1ABC9C", "rgba(26, 188, 156, 0.15)"),
            'v3': ("#0E6251", "rgba(14, 98, 81, 0.2)")
        }
        add_split_hist_metric(fig, df_combined, 'ce_loss', "CE Loss", ce_colors, row=7, col=2)

        # ==========================================
        # 格式與樣式設定
        # ==========================================
        fig.update_layout(
            height=2400, title_x=0.5, template="simple_white",
            font=dict(family="Times New Roman, Times, serif", size=14, color="black"),
            hovermode="x unified",
            legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1, bordercolor="black", borderwidth=1)
        )
        
        # X 軸設定 (Current Step)
        for r in range(1, 5):
            for c in range(1, 3):
                fig.update_xaxes(showline=True, linewidth=1.5, linecolor='black', mirror=True, gridcolor='rgba(200,200,200,0.3)', ticks="outside", tickwidth=1.5, ticklen=6, tickcolor='black', title_text="Current Step", row=r, col=c)
        fig.update_xaxes(showline=True, linewidth=1.5, linecolor='black', mirror=True, gridcolor='rgba(200,200,200,0.3)', ticks="outside", tickwidth=1.5, ticklen=6, tickcolor='black', title_text="Current Step", row=5, col=1)

        # X 軸設定 (Total Tokens Trained)
        fig.update_xaxes(showline=True, linewidth=1.5, linecolor='black', mirror=True, gridcolor='rgba(200,200,200,0.3)', ticks="outside", tickwidth=1.5, ticklen=6, tickcolor='black', title_text="Total Tokens Trained", row=6, col=1)
        for c in range(1, 3):
            fig.update_xaxes(showline=True, linewidth=1.5, linecolor='black', mirror=True, gridcolor='rgba(200,200,200,0.3)', ticks="outside", tickwidth=1.5, ticklen=6, tickcolor='black', title_text="Total Tokens Trained", row=7, col=c)
        
        # Y 軸設定
        fig.update_yaxes(showline=True, linewidth=1.5, linecolor='black', mirror=True, gridcolor='rgba(200,200,200,0.3)', ticks="outside", tickwidth=1.5, ticklen=6, tickcolor='black')
        
        for annotation in fig['layout']['annotations']: 
            annotation['font'] = dict(size=16, family="Times New Roman, Times, serif")
            if "Historical" in annotation['text']:
                annotation['text'] = f"<b>{annotation['text']}</b>"

        fig.show()
        if export_html:
            fig.write_html(output_filename)
            print(f"✅ Report saved: {output_filename}")

    def print_summary(self):
        last_curr = self.df_curr.iloc[-1]
        last_comb = self.df_combined.iloc[-1]
        
        print(f"\n{'='*65}")
        print(f"         MAMBA-3-XR LATEST STATUS (v3)")
        print(f"   (Current Step {int(last_curr['step'])} | Absolute Step {int(last_comb['abs_step'])})")
        print(f"{'='*65}")
        print(f"Time Elapsed (Total): {last_comb['cum_time_hrs']:.2f} Hours")
        print(f"Total Tokens Trained: {last_comb['total_tokens']:.2e} Tokens")
        print(f"Throughput          : {last_curr['tokens_per_sec_ma']:.2f} Tokens/s")
        print(f"Memory Usage        : {last_curr['mem_gb']:.2f} GB")
        print(f"Learning Rate       : {last_curr['lr']:.2e}")
        if 'grad_ma' in last_curr:
            print(f"Gradient Norm       : {last_curr['grad_ma']:.4f}")
        print(f"{'-'*65}")
        print(f"Total Loss          : {last_curr['total_loss_ma']:.4f}")
        print(f"PPL                 : {last_curr['ppl_ma']:.2f}")
        print(f"CE Loss             : {last_curr['ce_loss_ma']:.4f}")
        print(f"LB Loss             : {last_curr['lb_loss_ma']:.4f}")
        print(f"{'='*65}\n")

if __name__ == "__main__":
    V3_LOG_PATH = "/Users/hungwei/Desktop/Proj/Mamba3-XR/t4_training_log.csv" 
    V2_LOG_PATH = "/Users/hungwei/Desktop/Proj/Mamba3-XR/history/v2.csv"
    V1_LOG_PATH = "/Users/hungwei/Desktop/Proj/Mamba3-XR/history/v1.csv"
    
    try:
        # 參數皆已依據需求設定預設值
        analyzer = TrainingAnalyzer(
            v3_csv_path=V3_LOG_PATH, 
            v2_csv_path=V2_LOG_PATH, 
            v1_csv_path=V1_LOG_PATH
        )
        analyzer.generate_report()
        analyzer.print_summary()
    except Exception as e:
        print(f"❌ Error: {e}")

❌ Error: Error: Current v3 log file not found at /Users/hungwei/Desktop/Proj/Mamba3-XR/t4_training_log.csv
